# 卷積、池化與 CNN

對應原作業：Day 92–97。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 影像張量
PyTorch 影像維度是 `[N, C, H, W]`。`kernel_size=3, padding=1, stride=1` 維持高寬；`MaxPool2d(2)` 通常減半。以下用合成圖案測通完整流程，不需下載資料。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(42)
images = torch.randn(240, 1, 16, 16) * 0.15
labels = torch.arange(240) % 2
images[labels == 0, :, :, 5:7] += 1.0   # 垂直線
images[labels == 1, :, 5:7, :] += 1.0   # 水平線
train, valid = random_split(TensorDataset(images, labels), [192, 48],
                            generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train, batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid, batch_size=48, num_workers=0)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
                                      nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1),
                                      nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)))
        self.head = nn.Linear(16, 2)

    def forward(self, x):
        return self.head(torch.flatten(self.features(x), 1))

model = SmallCNN()
assert model(images[:4]).shape == (4, 2)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)


In [ ]:
for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.inference_mode():
    xb, yb = next(iter(valid_loader))
    accuracy = (model(xb).argmax(1) == yb).float().mean().item()
print("validation accuracy:", round(accuracy, 3))
assert accuracy > 0.8
